# 0 Download data from Kaggle





In [2]:
%pip install kagglehub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [kagglehub]
Note: you may need to restart the kernel to use updated packages.


In [3]:
import kagglehub
import pandas as pd

# Download latest version
# path = kagglehub.dataset_download("soumikrakshit/yahoo-answers-dataset")
# print("Path to dataset files:", path)

data = "train.csv"
df = pd.read_csv(data, header=None) # nrows=10000 nrows for number of samples

/opt/homebrew/Cellar/jupyterlab/4.5.2/libexec/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1 Exploratory Data Analysis

In [4]:
column_names = ['class_index', 'question_title', 'question_content', 'best_answer']
# Check the number of columns in the DataFrame `df`, not the string `data`
if len(df.columns) == 4:
    df.columns = column_names

print("--- Dataset Overview ---")
# Use df.info() for DataFrame information
print(df.info())
print("\n--- First 5 Rows ---")
print(df.head())

# 2. Missing Data Analysis
print("\n--- Missing Values Count ---")
# Check missing values in df
missing_values = df.isnull().sum()
print(missing_values)

--- Dataset Overview ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1400000 entries, 0 to 1399999
Data columns (total 4 columns):
 #   Column            Non-Null Count    Dtype 
---  ------            --------------    ----- 
 0   class_index       1400000 non-null  int64 
 1   question_title    1400000 non-null  object
 2   question_content  768311 non-null   object
 3   best_answer       1375404 non-null  object
dtypes: int64(1), object(3)
memory usage: 42.7+ MB
None

--- First 5 Rows ---
   class_index                                     question_title  \
0            5  why doesn't an optical mouse work on a glass t...   
1            6       What is the best off-road motorcycle trail ?   
2            3             What is Trans Fat? How to reduce that?   
3            7                         How many planes Fedex has?   
4            7  In the san francisco bay area, does it make se...   

                                    question_content  \
0                         

In [5]:
import plotly.graph_objects as go


# 2. Count samples per category
category_counts = df['class_index'].value_counts().sort_index()

# Define the exact mapping of class indices to their textual names
class_names = {
    1: 'Society & Culture',
    2: 'Science & Mathematics',
    3: 'Health',
    4: 'Education & Reference',
    5: 'Computers & Internet',
    6: 'Sports',
    7: 'Business & Finance',
    8: 'Entertainment & Music',
    9: 'Family & Relationships',
    10: 'Politics & Government'
}

# Rename the indices for display purposes only
display_counts = category_counts.rename(index=class_names)

# 3. Create bar chart
custom_colors = [
    '#667eea', '#764ba2', '#f093fb', '#4facfe', '#43e97b',
    '#f6d365', '#fda085', '#a18cd1', '#fbc2eb', '#84fab0'
]

fig = go.Figure(data=[go.Bar(
    x=display_counts.index,
    y=display_counts.values,
    marker_color=custom_colors,
    text=display_counts.values,  # Displays the exact value on each bar
    textposition='auto'          # Automatically places text inside or outside the bar
)])

fig.update_layout(
    title='Category Distribution',
    xaxis_title='Topic Category',
    yaxis_title='Number of Samples',
    xaxis_tickangle=-45,         # Angles the labels
    width=900,
    height=600,
    margin=dict(b=120)
)

fig.show()

In [6]:
import pandas as pd
import plotly.graph_objects as go
from collections import Counter

# 1. Define Stop Words
stop_words = stop_words = {
    "a", "about", "above", "across", "after", "afterwards", "again", "against", "all", "almost", 
    "alone", "along", "already", "also", "although", "always", "am", "among", "amongst", "amoungst", 
    "amount", "an", "and", "another", "any", "anyhow", "anyone", "anything", "anyway", "anywhere", 
    "are", "around", "as", "at", "back", "be", "became", "because", "become", "becomes", "becoming", 
    "been", "before", "beforehand", "behind", "being", "below", "beside", "besides", "between", 
    "beyond", "bill", "both", "bottom", "but", "by", "call", "can", "cannot", "cant", "co", "con", 
    "could", "couldnt", "cry", "de", "describe", "detail", "do", "done", "down", "due", "during", 
    "each", "eg", "eight", "either", "eleven", "else", "elsewhere", "empty", "enough", "etc", "even", 
    "ever", "every", "everyone", "everything", "everywhere", "except", "few", "fifteen", "fify", 
    "fill", "find", "fire", "first", "five", "for", "former", "formerly", "forty", "found", "four", 
    "from", "front", "full", "further", "get", "give", "go", "had", "has", "hasnt", "have", "he", 
    "hence", "her", "here", "hereafter", "hereby", "herein", "hereupon", "hers", "herself", "him", 
    "himself", "his", "how", "however", "hundred", "i", "ie", "if", "in", "inc", "indeed", "interest", 
    "into", "is", "it", "its", "itself", "keep", "last", "latter", "latterly", "least", "less", "ltd", 
    "made", "many", "may", "me", "meanwhile", "might", "mill", "mine", "more", "moreover", "most", 
    "mostly", "move", "much", "must", "my", "myself", "name", "namely", "neither", "never", 
    "nevertheless", "next", "nine", "no", "nobody", "none", "noone", "nor", "not", "nothing", "now", 
    "nowhere", "of", "off", "often", "on", "once", "one", "only", "onto", "or", "other", "others", 
    "otherwise", "our", "ours", "ourselves", "out", "over", "own", "part", "per", "perhaps", "please", 
    "put", "rather", "re", "same", "see", "seem", "seemed", "seeming", "seems", "serious", "several", 
    "she", "should", "show", "side", "since", "sincere", "six", "sixty", "so", "some", "somehow", 
    "someone", "something", "sometime", "sometimes", "somewhere", "still", "such", "system", "take", 
    "ten", "than", "that", "the", "their", "them", "themselves", "then", "thence", "there", "thereafter", 
    "thereby", "therefore", "therein", "thereupon", "these", "they", "thick", "thin", "third", "this", 
    "those", "though", "three", "through", "throughout", "thru", "thus", "to", "together", "too", "top", 
    "toward", "towards", "twelve", "twenty", "two", "un", "under", "until", "up", "upon", "us", "very", 
    "via", "was", "we", "well", "were", "what", "whatever", "when", "whence", "whenever", "where", 
    "whereafter", "whereas", "whereby", "wherein", "whereupon", "wherever", "whether", "which", "while", 
    "whither", "who", "whoever", "whole", "whom", "whose", "why", "will", "with", "within", "without", 
    "would", "yet", "you", "your", "yours", "yourself", "yourselves"
}

# 3. Prepare Text Data
# Handle NaN values to prevent errors during string concatenation
df.fillna('', inplace=True)
# Combine all text fields to represent the full document
df['full_text'] = df['question_title'] + " " + df['question_content'] + " " + df['best_answer']

# 4. Count Stop Words
print("Processing text corpus...")
all_words = ' '.join(df['full_text']).lower().split()
stop_word_counts = Counter([word for word in all_words if word in stop_words])
top_stop_words = stop_word_counts.most_common(20)

words = [word for word, count in top_stop_words]
counts = [count for word, count in top_stop_words]

# 5. Create Bar Chart
fig = go.Figure(data=[go.Bar(
    x=words,
    y=counts,
    marker_color='#764ba2',
    text=counts,
    textposition='outside'
)])

fig.update_layout(
    title='Top 20 Stop Words in Yahoo! Answers Dataset',
    xaxis_title='Stop Words',
    yaxis_title='Frequency',
    width=900,
    height=400,
    showlegend=False
)

fig.show()

# 6. Print Statistics
total_words = len(all_words)
total_stop_words = sum(stop_word_counts.values())
print("--- Stop Word Statistics ---")
print(f"Total words: {total_words:,}")
print(f"Stop words: {total_stop_words:,} ({total_stop_words/total_words*100:.1f}%)")
print(f"Unique stop words found: {len(stop_word_counts)}")

Processing text corpus...


--- Stop Word Statistics ---
Total words: 128,189,357
Stop words: 64,416,436 (50.3%)
Unique stop words found: 316


In [7]:
import plotly.graph_objects as go
import numpy as np

# ==========================================
# PART 1: Word Count Analysis
# ==========================================
print("--- Word Count Analysis ---")

# Calculate word counts
df['word_count'] = df['full_text'].str.split().str.len()

# Create histogram bins
# Note: Bin size is set to 100. If the maximum word count in Yahoo! Answers is very high, 
# this will create many bins, but it adheres to your template.
bins_word = np.arange(0, df['word_count'].max() + 50, 50)
hist_word, bin_edges_word = np.histogram(df['word_count'], bins=bins_word)

# Format bin labels
bin_labels_word = [f"{int(bin_edges_word[i])}-{int(bin_edges_word[i+1])}" for i in range(len(hist_word))]

# Create histogram
fig_word = go.Figure(data=[go.Bar(
    x=bin_labels_word,
    y=hist_word,
    marker_color='#667eea',
    text=hist_word,
    textposition='outside'
)])

fig_word.update_layout(
    title='Word Count Distribution (Yahoo! Answers)',
    xaxis_title='Word Count Ranges',
    yaxis_title='Number of Samples',
    width=900,
    height=400,
    showlegend=False,
    xaxis={'tickangle': -45}
)

fig_word.show()

# Print statistics
print(f"Mean: {df['word_count'].mean():.2f}")
print(f"Median: {df['word_count'].median():.2f}")
print(f"Min: {df['word_count'].min()}")
print(f"Max: {df['word_count'].max()}")
print("\nPercentiles:")
for p in [25, 50, 75, 90, 95, 99]:
    print(f"  {p}th: {df['word_count'].quantile(p/100):.0f}")


# ==========================================
# PART 2: Character Count Analysis
# ==========================================
print("\n--- Character Count Analysis ---")

# Calculate character counts
df['char_count'] = df['full_text'].str.len()

# Create histogram bins (bin size of 500)
bins_char = np.arange(0, df['char_count'].max() + 250, 250)
hist_char, bin_edges_char = np.histogram(df['char_count'], bins=bins_char)

bin_labels_char = [f"{int(bin_edges_char[i])}-{int(bin_edges_char[i+1])}" for i in range(len(hist_char))]

# Create histogram
fig_char = go.Figure(data=[go.Bar(
    x=bin_labels_char,
    y=hist_char,
    marker_color="#21c334",
    text=hist_char,
    textposition='outside'
)])

fig_char.update_layout(
    title='Character Count Distribution (Yahoo! Answers)',
    xaxis_title='Character Count Ranges',
    yaxis_title='Number of Samples',
    width=900,
    height=400,
    showlegend=False,
    xaxis={'tickangle': -45}
)

fig_char.show()

# Print statistics
print(f"Mean: {df['char_count'].mean():.2f}")
print(f"Median: {df['char_count'].median():.2f}")
print(f"Min: {df['char_count'].min()}")
print(f"Max: {df['char_count'].max()}")

# Estimate average word length
# Reusing the sum from the previously calculated 'word_count' column for efficiency
avg_word_length = df['char_count'].sum() / df['word_count'].sum()
print(f"\nAverage word length: {avg_word_length:.2f} characters")

--- Word Count Analysis ---


Mean: 91.56
Median: 60.00
Min: 1
Max: 1410

Percentiles:
  25th: 31
  50th: 60
  75th: 114
  90th: 199
  95th: 273
  99th: 537

--- Character Count Analysis ---


Mean: 520.76
Median: 339.00
Min: 12
Max: 7984

Average word length: 5.69 characters


Remove Stopword

In [8]:
import re
from collections import Counter
import plotly.graph_objects as go

print("Cleaning text and removing stopwords")

# 1. Define the Cleaning Function
def clean_and_remove_stopwords(text):
    # Convert to lowercase
    text = str(text).lower()
    # Remove all punctuation using regex (keeps only alphanumeric and whitespace)
    text = re.sub(r'[^\w\s]', '', text)
    # Tokenize by splitting on whitespace
    words = text.split()
    # Keep only alphabetical words that are NOT in our stop_words set
    cleaned_words = [w for w in words if w not in stop_words and w.isalpha()]
    return ' '.join(cleaned_words)

# 2. Apply Function to Create a Cleaned Column
df['text_no_stopwords'] = df['full_text'].apply(clean_and_remove_stopwords)

# Calculate the new word lengths per document for comparison
df['word_count_clean'] = df['text_no_stopwords'].str.split().str.len()

# 3. Analyze Top Meaningful Words
all_clean_words = ' '.join(df['text_no_stopwords']).split()
clean_word_counts = Counter(all_clean_words)
top_clean_words = clean_word_counts.most_common(20)

words_clean = [word for word, count in top_clean_words]
counts_clean = [count for word, count in top_clean_words]

# 4. Create Bar Chart for Meaningful Words
fig_clean = go.Figure(data=[go.Bar(
    x=words_clean,
    y=counts_clean,
    marker_color='#4facfe',  # A distinct blue to differentiate from previous charts
    text=counts_clean,
    textposition='outside'
)])

fig_clean.update_layout(
    title='Top 20 Most Frequent Words (Stopwords Removed)',
    xaxis_title='Meaningful Words',
    yaxis_title='Frequency',
    width=900,
    height=400,
    showlegend=False,
    xaxis={'tickangle': -45}
)

fig_clean.show()

# 5. Print Comparative Statistics
# Note: This assumes df['word_count'] exists from the previous execution block
if 'word_count' in df.columns:
    total_original_words = df['word_count'].sum()
else:
    # Fallback just in case
    total_original_words = sum(len(str(t).split()) for t in df['full_text'])

total_clean_words = len(all_clean_words)
removed_words = total_original_words - total_clean_words

print("\n--- Post-Cleaning Statistics ---")
print(f"Original total words: {total_original_words:,}")
print(f"Words remaining: {total_clean_words:,}")
print(f"Words removed: {removed_words:,} ({(removed_words/total_original_words)*100:.1f}%)")
print(f"Unique meaningful words (Vocabulary Size): {len(clean_word_counts):,}")
print(f"Average words per sample (Before): {df['word_count'].mean():.2f}")
print(f"Average words per sample (After): {df['word_count_clean'].mean():.2f}")

Cleaning text and removing stopwords



--- Post-Cleaning Statistics ---
Original total words: 128,189,357
Words remaining: 57,258,160
Words removed: 70,931,197 (55.3%)
Unique meaningful words (Vocabulary Size): 2,162,990
Average words per sample (Before): 91.56
Average words per sample (After): 40.90


In [9]:
import pandas as pd
import plotly.graph_objects as go
import re

print("Calculating vocabulary richness per category. This requires processing the entire corpus...")

# 1. Define Class Names Mapping (Re-declaring for completeness)
class_names = {
    1: 'Society & Culture',
    2: 'Science & Mathematics',
    3: 'Health',
    4: 'Education & Reference',
    5: 'Computers & Internet',
    6: 'Sports',
    7: 'Business & Finance',
    8: 'Entertainment & Music',
    9: 'Family & Relationships',
    10: 'Politics & Government'
}

# 2. Calculate Vocabulary Richness per Category
vocab_stats = []

for class_idx in sorted(df['class_index'].unique()):
    # Isolate data for the current category
    cat_df = df[df['class_index'] == class_idx]
    category_name = class_names.get(class_idx, f"Class {class_idx}")
    
    # Concatenate all text and convert to lowercase
    all_text = ' '.join(cat_df['full_text']).lower()
    
    # Use regular expression to extract words with 3 or more characters
    words = re.findall(r'\b[a-z]{3,}\b', all_text)
    
    # Filter out stopwords (utilizing the set defined in the previous step)
    words = [w for w in words if w not in stop_words]
    
    unique_words = len(set(words))
    total_words = len(words)
    
    # Calculate Type-Token Ratio (TTR)
    ttr = unique_words / total_words if total_words > 0 else 0
    
    vocab_stats.append({
        'Category': category_name,
        'Unique Words': unique_words,
        'Total Words': total_words,
        'TTR': round(ttr, 4),
        'Samples': len(cat_df)
    })

# Convert the results into a structured DataFrame
vocab_df = pd.DataFrame(vocab_stats)

# 3. Create Bar Chart
fig = go.Figure()

fig.add_trace(go.Bar(
    x=vocab_df['Category'],
    y=vocab_df['Unique Words'],
    name='Unique Words',
    marker_color='#4facfe',
    text=vocab_df['Unique Words'],  # Display the exact integer on the bar
    textposition='auto'
))

# 4. Layout Optimization
fig.update_layout(
    title='Vocabulary Richness by Category (Yahoo! Answers)',
    xaxis_title='Topic Category',
    yaxis_title='Unique Words Count',
    width=900,
    height=550,
    xaxis={'tickangle': -45},
    margin=dict(b=120)  # Margin to accommodate angled labels
)

fig.show()

# 5. Print Tabular Statistics
print("\n--- Vocabulary Richness Statistics ---")
print(vocab_df.to_string(index=False))

Calculating vocabulary richness per category. This requires processing the entire corpus...



--- Vocabulary Richness Statistics ---
              Category  Unique Words  Total Words    TTR  Samples
     Society & Culture        179621      6765800 0.0265   140000
 Science & Mathematics        182383      6365025 0.0287   140000
                Health        147638      6825650 0.0216   140000
 Education & Reference        206346      5612358 0.0368   140000
  Computers & Internet        131064      5576682 0.0235   140000
                Sports        153819      4761439 0.0323   140000
    Business & Finance        151676      5333581 0.0284   140000
 Entertainment & Music        158597      4564118 0.0347   140000
Family & Relationships        107733      5566194 0.0194   140000
 Politics & Government        152525      6853533 0.0223   140000


In [10]:
import plotly.graph_objects as go
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

print("Sampling data to optimize computation time...")

# 1. Stratified Sampling
# Select N samples per category and reset the index for sparse matrix compatibility
df_sampled = df.groupby('class_index').sample(n=4000, random_state=42).reset_index(drop=True)

# 2. Define Class Names Mapping
class_names = {
    1: 'Society & Culture',
    2: 'Science & Mathematics',
    3: 'Health',
    4: 'Education & Reference',
    5: 'Computers & Internet',
    6: 'Sports',
    7: 'Business & Finance',
    8: 'Entertainment & Music',
    9: 'Family & Relationships',
    10: 'Politics & Government'
}

# 3. Vectorization Parameters
categories = sorted(df_sampled['class_index'].unique())
category_labels = [class_names.get(c, f"Class {c}") for c in categories]

# Initialize TF-IDF Vectorizer
vectorizer = TfidfVectorizer(max_features=5000, stop_words=list(stop_words))
tfidf_matrix = vectorizer.fit_transform(df_sampled['full_text'])

# 4. Compute Similarity Matrix
print("Computing TF-IDF matrices and Cosine Similarities on the sampled data...")
n_cats = len(categories)
similarity_matrix = np.zeros((n_cats, n_cats))

for i, cat1 in enumerate(categories):
    # Extract row indices for the first category using the new zero-indexed dataframe
    cat1_indices = df_sampled[df_sampled['class_index'] == cat1].index
    cat1_vectors = tfidf_matrix[cat1_indices]
    
    for j, cat2 in enumerate(categories):
        # Extract row indices for the second category
        cat2_indices = df_sampled[df_sampled['class_index'] == cat2].index
        cat2_vectors = tfidf_matrix[cat2_indices]
        
        # Compute pairwise cosine similarities
        pairwise_sim = cosine_similarity(cat1_vectors, cat2_vectors)
        
        # Calculate the mean similarity score
        similarity_matrix[i, j] = np.mean(pairwise_sim)

# 5. Generate Heatmap Visualization
fig = go.Figure(data=go.Heatmap(
    z=similarity_matrix,
    x=category_labels,
    y=category_labels,
    colorscale='Purples',
    text=similarity_matrix,
    texttemplate='%{text:.3f}',
    textfont={"size": 10},
    colorbar=dict(title="Mean Cosine Sim")
))

fig.update_layout(
    title='Category Similarity Matrix (Sampled Data)',
    xaxis_title='Topic Category',
    yaxis_title='Topic Category',
    width=900,
    height=800,
    yaxis={'autorange': 'reversed'}, 
    xaxis={'tickangle': -45},
    margin=dict(l=150, b=150)
)

fig.show()

# 6. Output Statistical Summary
print("\n--- Category Similarity Matrix ---")
for i, cat1 in enumerate(categories):
    label1 = category_labels[i]
    print(f"{label1:25s}: {similarity_matrix[i][i]:.3f} (within-category)")
    for j, cat2 in enumerate(categories):
        if i < j:
            label2 = category_labels[j]
            sim = similarity_matrix[i][j]
            status = "High overlap" if sim > 0.02 else "✓ Distinct"
            print(f"  vs {label2:25s}: {sim:.3f} | {status}")

Sampling data to optimize computation time...
Computing TF-IDF matrices and Cosine Similarities on the sampled data...



--- Category Similarity Matrix ---
Society & Culture        : 0.020 (within-category)
  vs Science & Mathematics    : 0.008 | ✓ Distinct
  vs Health                   : 0.013 | ✓ Distinct
  vs Education & Reference    : 0.011 | ✓ Distinct
  vs Computers & Internet     : 0.009 | ✓ Distinct
  vs Sports                   : 0.009 | ✓ Distinct
  vs Business & Finance       : 0.012 | ✓ Distinct
  vs Entertainment & Music    : 0.012 | ✓ Distinct
  vs Family & Relationships   : 0.019 | ✓ Distinct
  vs Politics & Government    : 0.013 | ✓ Distinct
Science & Mathematics    : 0.011 (within-category)
  vs Health                   : 0.009 | ✓ Distinct
  vs Education & Reference    : 0.008 | ✓ Distinct
  vs Computers & Internet     : 0.007 | ✓ Distinct
  vs Sports                   : 0.006 | ✓ Distinct
  vs Business & Finance       : 0.007 | ✓ Distinct
  vs Entertainment & Music    : 0.007 | ✓ Distinct
  vs Family & Relationships   : 0.009 | ✓ Distinct
  vs Politics & Government    : 0.007 | ✓ Dist

# 2 Data Preprocessing

Shift labels

In [11]:
import re

# Shift labels from 1-10 to 0-9
df['label'] = df['class_index'] - 1

print(f"Original unique labels: {sorted(df['class_index'].unique())}")
print(f"Zero-indexed unique labels: {sorted(df['label'].unique())}")

Original unique labels: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]
Zero-indexed unique labels: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]


Text cleaning

In [12]:
def clean_text(text):
    if not isinstance(text, str):
        return ""

    # Convert text to lowercase
    text = text.lower()

    # Replace literal '\n', '\r', '\t' characters with a space
    text = text.replace('\\n', ' ').replace('\\r', ' ').replace('\\t', ' ')

    # Handle actual newlines just in case they exist
    text = text.replace('\n', ' ').replace('\r', ' ')

    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # Remove HTML tags
    text = re.sub(r'<.*?>', ' ', text)

    # Remove irregular characters
    text = re.sub(r"[^a-z0-9\s.,?!'\"]", ' ', text)

    # 7. Normalize whitespace (collapse multiple spaces into one)
    text = re.sub(r'\s+', ' ', text).strip()

    return text


df['cleaned_text'] = df['full_text'].apply(clean_text)

# Verify
print("\n--- Cleaning Check ---")
sample_idx = df.sample(1).index[0]
print("ORIGINAL TEXT:\n", df.loc[sample_idx, 'full_text'][:300])
print("\nCLEANED TEXT:\n", df.loc[sample_idx, 'cleaned_text'][:300])


--- Cleaning Check ---
ORIGINAL TEXT:
 2/30, net 60 means?  If it's terms on a bill, it means you can take a 2% discount if you pay within 30 days and the bill must be paid by 60 days.

CLEANED TEXT:
 2 30, net 60 means? if it's terms on a bill, it means you can take a 2 discount if you pay within 30 days and the bill must be paid by 60 days.


Tokenizing for LSTM

In [13]:
from collections import Counter
import pandas as pd

# Define critical special tokens and their reserved indices
PAD_TOKEN = "<PAD>"  # Used later to make sequence lengths uniform in a batch
UNK_TOKEN = "<UNK>"  # Used for words that fall outside our vocabulary
PAD_IDX = 0
UNK_IDX = 1

# Limit vocabulary size to prevent the embedding layer from consuming excessive memory
# 25,000 to 50,000 is typical for this dataset.
MAX_VOCAB_SIZE = 50000

def build_vocabulary(text_series, max_size):
    """
    Scans the corpus, calculates word frequencies, and constructs a mapping
    from words to integer indices.
    """
    print("Scanning corpus to build vocabulary...")
    word_counts = Counter()

    # Iterate through all cleaned text and count word frequencies
    for text in text_series:
        if isinstance(text, str):
            word_counts.update(text.split())

    # Keep only the most common words up to max_size
    most_common = word_counts.most_common(max_size)

    # Initialize the dictionaries with our special tokens
    word2idx = {PAD_TOKEN: PAD_IDX, UNK_TOKEN: UNK_IDX}
    idx2word = {PAD_IDX: PAD_TOKEN, UNK_IDX: UNK_TOKEN}

    # Assign an integer index to each word, starting from 2
    for idx, (word, _) in enumerate(most_common, start=2):
        word2idx[word] = idx
        idx2word[idx] = word

    print(f"Vocabulary built. Total unique words retained: {len(word2idx)}")
    return word2idx, idx2word

def encode_sequence(text, word2idx):
    """
    Converts a raw string into a list of integers based on the vocabulary.
    Words not in the vocabulary are assigned the <UNK> index.
    """
    if not isinstance(text, str):
        return []

    return [word2idx.get(word, UNK_IDX) for word in text.split()]

# Build the vocabulary using the cleaned text column
word2idx, idx2word = build_vocabulary(df['cleaned_text'], MAX_VOCAB_SIZE)

# Encode the dataset
print("Encoding text sequences to integers...")
df['encoded_sequence'] = df['cleaned_text'].apply(lambda x: encode_sequence(x, word2idx))

# Verify
vocab_size = len(word2idx)
print(f"Vocabulary size is: {vocab_size}")

sample_idx = df.sample(1).index[0]
print("\n--- Tokenization Verification ---")
print("CLEANED TEXT:   ", df.loc[sample_idx, 'cleaned_text'][:100], "...")
print("ENCODED ARRAYS: ", df.loc[sample_idx, 'encoded_sequence'][:15], "...")

Scanning corpus to build vocabulary...
Vocabulary built. Total unique words retained: 50002
Encoding text sequences to integers...
Vocabulary size is: 50002

--- Tokenization Verification ---
CLEANED TEXT:    what was the result of the ll bean and nordstrom lawsuit? l.l. bean settled the case with claria and ...
ENCODED ARRAYS:  [22, 33, 2, 916, 7, 2, 3709, 12460, 4, 1, 47521, 1, 12460, 7765, 2] ...


Tokenizing for Transformer

In [14]:
from transformers import AutoTokenizer

data_1 = df #clone data

# Define the model checkpoint
MODEL_CHECKPOINT = "bert-base-uncased"

print(f"Loading pre-trained tokenizer for: {MODEL_CHECKPOINT}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

# Define the maximum sequence length based on your earlier EDA (e.g., 99th percentile)
# 256 or 512 are standard lengths for BERT.
MAX_LEN = 256

def tokenize_for_transformer(text_series):
    """
    Applies the Hugging Face tokenizer to a pandas Series of text.
    Returns the input IDs and Attention Masks.
    """
    # The tokenizer handles truncation, special tokens ([CLS], [SEP]),
    # and mapping to the pre-trained vocabulary.
    # Note: We do NOT pad here. We will pad dynamically in the DataLoader.
    encoded_batch = tokenizer(
        text_series.tolist(),
        truncation=True,
        max_length=MAX_LEN,
        padding=False, # Dynamic padding is more efficient
        return_attention_mask=True
    )

    return encoded_batch['input_ids'], encoded_batch['attention_mask']

# Test the tokenizer on a small sample to verify
sample_text = data_1['cleaned_text'].iloc[0:2]
sample_ids, sample_masks = tokenize_for_transformer(sample_text)

print("\n--- Transformer Tokenization Verification ---")
print("ORIGINAL: ", sample_text.iloc[0][:100], "...")
print("TOKENS:   ", tokenizer.convert_ids_to_tokens(sample_ids[0])[:15])
print("INPUT IDS:", sample_ids[0][:15])

Loading pre-trained tokenizer for: bert-base-uncased

--- Transformer Tokenization Verification ---
ORIGINAL:  why doesn't an optical mouse work on a glass table? or even on some surfaces? optical mice use an le ...
TOKENS:    ['[CLS]', 'why', 'doesn', "'", 't', 'an', 'optical', 'mouse', 'work', 'on', 'a', 'glass', 'table', '?', 'or']
INPUT IDS: [101, 2339, 2987, 1005, 1056, 2019, 9380, 8000, 2147, 2006, 1037, 3221, 2795, 1029, 2030]


In [16]:
df.columns

Index(['class_index', 'question_title', 'question_content', 'best_answer',
       'full_text', 'word_count', 'char_count', 'text_no_stopwords',
       'word_count_clean', 'label', 'cleaned_text', 'encoded_sequence'],
      dtype='object')

In [17]:
# --- ADD THIS AFTER YOUR MEAN SENTENCE LENGTH ANALYSIS ---

print("\n--- 9. Stop Words Analysis (Raw Data) ---")
# Calculate total words first
total_words_corpus = df['word_count'].sum()

# Function to count stopwords in a single text
def count_stopwords(text):
    words = str(text).lower().split()
    return sum(1 for w in words if w in stop_words)

df['stopword_count'] = df['full_text'].apply(count_stopwords)
total_stopwords = df['stopword_count'].sum()
stopword_percentage = (total_stopwords / total_words_corpus) * 100 if total_words_corpus > 0 else 0

print(f"Total Words in Corpus: {total_words_corpus:,}")
print(f"Total Stop Words: {total_stopwords:,}")
print(f"Percentage of Stop Words: {stopword_percentage:.2f}%\n")

# --- 10. Create Clean Text Column for Semantic Analysis ---
import re

def remove_stopwords_and_punct(text):
    text = str(text).lower()
    # Remove basic punctuation
    text = re.sub(r'[^\w\s]', '', text)
    words = text.split()
    # Keep only non-stopwords
    cleaned_words = [w for w in words if w not in stop_words and w.isalpha()]
    return ' '.join(cleaned_words)

print("Processing text to remove stopwords for semantic analysis...")
df['text_no_stopwords'] = df['full_text'].apply(remove_stopwords_and_punct)

# --- 11. Category Keywords & Vocabulary Richness ---
print("\n--- Vocabulary Richness & Category Keywords ---")

# Group the cleaned text by class index
category_groups = df.groupby('class_index')

vocab_data = []

for class_idx, group in category_groups:
    # Combine all text for this category into one massive string
    combined_text = ' '.join(group['text_no_stopwords'])
    words = combined_text.split()

    total_words = len(words)
    unique_words = len(set(words))

    # Store for tabular display
    vocab_data.append({
        'Category/Class': class_idx,
        'Total Words': total_words,
        'Unique Words': unique_words,
        'Articles/Samples': len(group)
    })

    # Get top 5 keywords for this category
    top_keywords = Counter(words).most_common(5)
    print(f"Class {class_idx} Top Keywords: {[word for word, count in top_keywords]}")

# Display Vocabulary Richness Table
vocab_df = pd.DataFrame(vocab_data)
print("\nVocabulary Richness Table:")
print(vocab_df.to_string(index=False))


--- 9. Stop Words Analysis (Raw Data) ---
Total Words in Corpus: 128,189,357
Total Stop Words: 64,416,436
Percentage of Stop Words: 50.25%

Processing text to remove stopwords for semantic analysis...

--- Vocabulary Richness & Category Keywords ---
Class 1 Top Keywords: ['people', 'god', 'like', 'just', 'dont']
Class 2 Top Keywords: ['water', 'like', 'know', 'does', 'time']
Class 3 Top Keywords: ['like', 'just', 'dont', 'know', 'help']
Class 4 Top Keywords: ['school', 'know', 'like', 'help', 'need']
Class 5 Top Keywords: ['computer', 'use', 'need', 'want', 'like']
Class 6 Top Keywords: ['team', 'think', 'like', 'world', 'good']
Class 7 Top Keywords: ['like', 'know', 'dont', 'just', 'want']
Class 8 Top Keywords: ['like', 'know', 'think', 'song', 'just']
Class 9 Top Keywords: ['like', 'just', 'dont', 'know', 'love']
Class 10 Top Keywords: ['people', 'think', 'dont', 'just', 'like']

Vocabulary Richness Table:
 Category/Class  Total Words  Unique Words  Articles/Samples
              1 

# 3 Models

## 3A LSTM Block

In [18]:
import torch
import torch.nn as nn

class myLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers,
                 bidirectional, dropout, pad_idx):
        """
        Initializes the Bidirectional LSTM model.
        """
        super().__init__()

        # 1. Embedding Layer: Converts integer token IDs into dense vectors.
        # padding_idx=pad_idx ensures the <PAD> token does not contribute to the gradient.
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)

        # 2. LSTM Layer: Processes the sequence of embeddings.
        # batch_first=True ensures input tensors have the shape (batch_size, seq_length, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim,
                            hidden_dim,
                            num_layers=n_layers,
                            bidirectional=bidirectional,
                            dropout=dropout if n_layers > 1 else 0,
                            batch_first=True)

        # 3. Fully Connected (Linear) Classifier Layer
        # If bidirectional, the hidden state size is doubled because we concatenate
        # the final forward and backward hidden states.
        self.fc = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, output_dim)

        # 4. Dropout Layer: Applied to the final hidden state for regularization
        self.dropout = nn.Dropout(dropout)

    def forward(self, text, text_lengths):
        """
        Defines the forward pass of the model.
        """
        # text shape: (batch_size, seq_length)

        # 1. Pass through embedding layer
        # embedded shape: (batch_size, seq_length, embedding_dim)
        embedded = self.embedding(text)

        # 2. Pass through LSTM layer
        # output shape: (batch_size, seq_length, hidden_dim * num_directions)
        # hidden shape: (num_layers * num_directions, batch_size, hidden_dim)
        output, (hidden, cell) = self.lstm(embedded)

        # 3. Extract the final hidden state
        # We need the final hidden state of the top layer.
        # If bidirectional, we concatenate the final forward hidden state and final backward hidden state.
        if self.lstm.bidirectional:
            # hidden[-2, :, :] is the forward top layer
            # hidden[-1, :, :] is the backward top layer
            hidden_final = self.dropout(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1))
        else:
            hidden_final = self.dropout(hidden[-1,:,:])

        # hidden_final shape: (batch_size, hidden_dim * num_directions)

        # 4. Pass through the linear classifier
        # return shape: (batch_size, output_dim)
        return self.fc(hidden_final)

Usage

In [19]:
# Assuming vocab_size and PAD_IDX are defined from the previous tokenization step
# vocab_size = len(word2idx) # Approximately 30,002
# PAD_IDX = 0

INPUT_DIM = vocab_size
EMBEDDING_DIM = 128     # Dimensionality of the word vectors
HIDDEN_DIM = 256        # Number of features in the hidden state of the LSTM
OUTPUT_DIM = 10         # 10 classes
N_LAYERS = 2            # Number of stacked LSTM layers
BIDIRECTIONAL = True    # Use Bi-LSTM
DROPOUT = 0.5           # 50% probability of dropping a neuron to prevent overfitting

# Instantiate the model
lstm_model = myLSTM(
    vocab_size=INPUT_DIM,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    output_dim=OUTPUT_DIM,
    n_layers=N_LAYERS,
    bidirectional=BIDIRECTIONAL,
    dropout=DROPOUT,
    pad_idx=PAD_IDX
)

print(lstm_model)

# Function to calculate the number of trainable parameters (useful for your report)
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'The model has {count_parameters(lstm_model):,} trainable parameters')

myLSTM(
  (embedding): Embedding(50002, 128, padding_idx=0)
  (lstm): LSTM(128, 256, num_layers=2, batch_first=True, dropout=0.5, bidirectional=True)
  (fc): Linear(in_features=512, out_features=10, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)
The model has 8,772,874 trainable parameters


## 3B Transformer Block

In [20]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoConfig

class myTransformer(nn.Module):
    def __init__(self, checkpoint, num_classes, dropout_rate=0.3, freeze_backbone=False):
        """
        Initializes the Transformer classification model.
        """
        super().__init__()

        # 1. Load the pre-trained Transformer backbone and its configuration
        self.config = AutoConfig.from_pretrained(checkpoint)
        self.bert = AutoModel.from_pretrained(checkpoint)

        # 2. Optional Fine-Tuning Strategy (Assignment Extension Criterion)
        # Freezing the backbone drastically reduces training time and memory usage.
        if freeze_backbone:
            for param in self.bert.parameters():
                param.requires_grad = False

        # 3. Regularization
        # We apply dropout to the pooled output before the final classification layer.
        self.dropout = nn.Dropout(dropout_rate)

        # 4. Classification Head
        # BERT outputs a hidden state of size 768 (for bert-base-uncased).
        # We project this down to our 10 target classes.
        self.fc = nn.Linear(self.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        """
        Defines the forward pass.
        """
        # Pass the inputs through the Transformer backbone
        # outputs is a complex object containing various hidden states
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )

        # Extract the "pooler_output".
        # In BERT, this is the representation of the [CLS] token, which has been
        # further processed by a linear layer and a Tanh activation function.
        # It serves as an aggregated representation of the entire sequence.
        pooled_output = outputs.pooler_output

        # Pass through dropout and the final linear classifier
        x = self.dropout(pooled_output)
        logits = self.fc(x)

        return logits

Usage



In [21]:
# Hyperparameters
TRANSFORMER_CHECKPOINT = "bert-base-uncased"
NUM_CLASSES = 10
DROPOUT_RATE = 0.3
FREEZE_BACKBONE = False # Set to True to only train the linear classification head

# Instantiate the model
transformer_model = myTransformer(
    checkpoint=TRANSFORMER_CHECKPOINT,
    num_classes=NUM_CLASSES,
    dropout_rate=DROPOUT_RATE,
    freeze_backbone=FREEZE_BACKBONE
)

print(transformer_model)

def count_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total

trainable_params, total_params = count_parameters(transformer_model)
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

myTransformer(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise

# 4 Training and Evaluation

This section trains and evaluates both models using standard classification metrics.

In [ ]:
import random
import numpy as np
import torch
from sklearn.model_selection import train_test_split

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Ensure required columns exist
required_cols = ["label", "cleaned_text", "encoded_sequence"]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in df: {missing_cols}")

# Train/Validation/Test split (80/10/10) with stratification
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=SEED,
    stratify=temp_df["label"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train size: {len(train_df):,}")
print(f"Val size:   {len(val_df):,}")
print(f"Test size:  {len(test_df):,}")

## 4A LSTM Training and Evaluation

In [ ]:
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

MAX_SEQ_LEN_LSTM = 256
BATCH_SIZE_LSTM = 64
EPOCHS_LSTM = 3
LR_LSTM = 1e-3

class LSTMDataset(Dataset):
    def __init__(self, frame):
        self.sequences = frame["encoded_sequence"].tolist()
        self.labels = frame["label"].tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        if not isinstance(seq, list):
            seq = []
        if len(seq) > MAX_SEQ_LEN_LSTM:
            seq = seq[:MAX_SEQ_LEN_LSTM]
        return torch.tensor(seq, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

def lstm_collate_fn(batch):
    sequences, labels = zip(*batch)
    lengths = torch.tensor([len(seq) for seq in sequences], dtype=torch.long)
    padded = pad_sequence(sequences, batch_first=True, padding_value=PAD_IDX)
    labels = torch.stack(labels)
    return padded, lengths, labels

def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

def run_lstm_epoch(model, dataloader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    all_preds, all_labels = [], []

    for padded, lengths, labels in dataloader:
        padded = padded.to(device)
        lengths = lengths.to(device)
        labels = labels.to(device)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            logits = model(padded, lengths)
            loss = criterion(logits, labels)

            if is_train:
                loss.backward()
                optimizer.step()

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / max(len(dataloader), 1)
    metrics = compute_metrics(all_labels, all_preds)
    return avg_loss, metrics

# Build data loaders
lstm_train_ds = LSTMDataset(train_df)
lstm_val_ds = LSTMDataset(val_df)
lstm_test_ds = LSTMDataset(test_df)

lstm_train_loader = DataLoader(lstm_train_ds, batch_size=BATCH_SIZE_LSTM, shuffle=True, collate_fn=lstm_collate_fn)
lstm_val_loader = DataLoader(lstm_val_ds, batch_size=BATCH_SIZE_LSTM, shuffle=False, collate_fn=lstm_collate_fn)
lstm_test_loader = DataLoader(lstm_test_ds, batch_size=BATCH_SIZE_LSTM, shuffle=False, collate_fn=lstm_collate_fn)

# Train
lstm_model = lstm_model.to(device)
lstm_criterion = torch.nn.CrossEntropyLoss()
lstm_optimizer = optim.Adam(lstm_model.parameters(), lr=LR_LSTM)

print("\n=== LSTM Training ===")
for epoch in range(1, EPOCHS_LSTM + 1):
    train_loss, train_metrics = run_lstm_epoch(lstm_model, lstm_train_loader, lstm_criterion, lstm_optimizer)
    val_loss, val_metrics = run_lstm_epoch(lstm_model, lstm_val_loader, lstm_criterion)

    print(
        f"Epoch {epoch}/{EPOCHS_LSTM} | "
        f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_metrics['accuracy']:.4f} | "
        f"Val Precision: {val_metrics['precision']:.4f} | "
        f"Val Recall: {val_metrics['recall']:.4f} | "
        f"Val F1: {val_metrics['f1']:.4f}"
    )

# Final test evaluation
test_loss, test_metrics = run_lstm_epoch(lstm_model, lstm_test_loader, lstm_criterion)
print("\n=== LSTM Test Metrics ===")
print(f"Test Loss:      {test_loss:.4f}")
print(f"Test Accuracy:  {test_metrics['accuracy']:.4f}")
print(f"Test Precision: {test_metrics['precision']:.4f}")
print(f"Test Recall:    {test_metrics['recall']:.4f}")
print(f"Test F1:        {test_metrics['f1']:.4f}")

## 4B Transformer Training and Evaluation

In [ ]:
from torch.utils.data import Dataset, DataLoader

BATCH_SIZE_TRANSFORMER = 16
EPOCHS_TRANSFORMER = 2
LR_TRANSFORMER = 2e-5
MAX_LEN_TRANSFORMER = 256

class TransformerDataset(Dataset):
    def __init__(self, frame):
        self.texts = frame["cleaned_text"].fillna("").astype(str).tolist()
        self.labels = frame["label"].tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.texts[idx], int(self.labels[idx])

def transformer_collate_fn(batch):
    texts, labels = zip(*batch)
    encoded = tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=MAX_LEN_TRANSFORMER,
        return_tensors="pt"
    )
    labels = torch.tensor(labels, dtype=torch.long)
    return encoded["input_ids"], encoded["attention_mask"], labels

def run_transformer_epoch(model, dataloader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    all_preds, all_labels = [], []

    for input_ids, attention_mask, labels in dataloader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(logits, labels)

            if is_train:
                loss.backward()
                optimizer.step()

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / max(len(dataloader), 1)
    metrics = compute_metrics(all_labels, all_preds)
    return avg_loss, metrics

# Build data loaders
tr_train_ds = TransformerDataset(train_df)
tr_val_ds = TransformerDataset(val_df)
tr_test_ds = TransformerDataset(test_df)

tr_train_loader = DataLoader(tr_train_ds, batch_size=BATCH_SIZE_TRANSFORMER, shuffle=True, collate_fn=transformer_collate_fn)
tr_val_loader = DataLoader(tr_val_ds, batch_size=BATCH_SIZE_TRANSFORMER, shuffle=False, collate_fn=transformer_collate_fn)
tr_test_loader = DataLoader(tr_test_ds, batch_size=BATCH_SIZE_TRANSFORMER, shuffle=False, collate_fn=transformer_collate_fn)

# Train
transformer_model = transformer_model.to(device)
tr_criterion = torch.nn.CrossEntropyLoss()
tr_optimizer = torch.optim.AdamW(transformer_model.parameters(), lr=LR_TRANSFORMER)

print("\n=== Transformer Training ===")
for epoch in range(1, EPOCHS_TRANSFORMER + 1):
    train_loss, train_metrics = run_transformer_epoch(transformer_model, tr_train_loader, tr_criterion, tr_optimizer)
    val_loss, val_metrics = run_transformer_epoch(transformer_model, tr_val_loader, tr_criterion)

    print(
        f"Epoch {epoch}/{EPOCHS_TRANSFORMER} | "
        f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_metrics['accuracy']:.4f} | "
        f"Val Precision: {val_metrics['precision']:.4f} | "
        f"Val Recall: {val_metrics['recall']:.4f} | "
        f"Val F1: {val_metrics['f1']:.4f}"
    )

# Final test evaluation
test_loss, test_metrics = run_transformer_epoch(transformer_model, tr_test_loader, tr_criterion)
print("\n=== Transformer Test Metrics ===")
print(f"Test Loss:      {test_loss:.4f}")
print(f"Test Accuracy:  {test_metrics['accuracy']:.4f}")
print(f"Test Precision: {test_metrics['precision']:.4f}")
print(f"Test Recall:    {test_metrics['recall']:.4f}")
print(f"Test F1:        {test_metrics['f1']:.4f}")